Решать будем задачу машинного перевода с английского на французский


В этой домашке будем делать RNN + Attention как на 55-60 слайдах этой [лекции](https://github.com/fintech-dl-hse/course/blob/f8d2a6209bf0002ec536be3fe202d8d795ee0d98/lectures/lecture8_seq2seq_nlp.pdf)

По большому счету, в этой домашке вам нужно будет
1. Правильно рассатвить параметры для подготовленного шаблона
2. Дополнить пропущеные куски кода в методах обучения модели
3. Переобучить модель на нескольких батчах

То, что модель переобучается на небольшом количестве данных -- это необходимый критерий того, что модель в принципе адекватная. Но не достаточный, тк при переобучении мы не можем оценить обобщающую способность модели. 

In [1]:
!pip install -q datasets pytorch-lightning transformers evaluate sacrebleu accelerate

# Загрузка датасета и предобработка

In [2]:
import torch
import torch.nn as nn

from typing import List, Optional

from datasets import load_dataset
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
from tokenizers.models import BPE

from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from transformers import PreTrainedTokenizerFast, PreTrainedTokenizer

In [3]:
books = load_dataset("opus_books", "en-fr")

books["train"] = books["train"].select(range(10000))
books = books.filter(lambda x: len(x['translation']['en']) < 250)
print(books)


  0%|          | 0/1 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 8919
    })
})


In [4]:
# посмотрим на пример, как выглядит единица данных в датасете
# это корпус параллельных предложений
books['train'][0]

{'id': '2520',
 'translation': {'en': 'Meaulnes, at the request of Mademoiselle de Galais, is now very nice to him.',
  'fr': 'Meaulnes, sur la prière de Mlle de Galais, est maintenant très aimable avec lui.'}}

In [5]:
# нейросети напрямую с текстовыми данными работать не могут, поэтому нужно преобразовать текстовые данные в числовые
# токенизатор отвечает за преобразование символьной последовательности (строки с текстом)
# в последовательность интов, с которыми уже может работать модель

# для обучения BPE токенизатора, надо подготовить файлик
# со всеми предложениями из нашего датасета

ALL_SETENCES_FILE = 'all_book_sentences.txt'

with open(ALL_SETENCES_FILE, 'w') as f:
    for item in books['train']:
        f.write(item['translation']['en'] + "\n")
        f.write(item['translation']['fr'] + "\n")

In [6]:
# проверим, что файлик правильно сохранился
!head all_book_sentences.txt

Meaulnes, at the request of Mademoiselle de Galais, is now very nice to him.
Meaulnes, sur la prière de Mlle de Galais, est maintenant très aimable avec lui.
Charlotte had seen them from her husband's room, crossing the road, and immediately running into the other, told the girls what an honour they might expect, adding:
Du bureau de son mari Charlotte les vit traverser la route et courut annoncer aux jeunes filles l’honneur qui leur était fait :
"La! my dear," said Maria, quite shocked at the mistake, "it is not Lady Catherine. The old lady is Mrs. Jenkinson, who lives with them; the other is Miss de Bourgh.
– Oh ! ma chere, dit Maria scandalisée de sa méprise, ce n’est pas lady Catherine, c’est miss Jenkins, la dame de compagnie, et miss de Bourgh.
Instead of receiving any such letter of excuse from his friend, as Elizabeth half expected Mr. Bingley to do, he was able to bring Darcy with him to Longbourn before many days had passed after Lady Catherine's visit.
Au lieu de recevoir de

In [7]:
# обученм токенизатор
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

VOCAB_SIZE = 20000

bpe_trainer = BpeTrainer(special_tokens=["[UNK]", "[BOS]", "[EOS]", "[PAD]"], show_progress=True, vocab_size=VOCAB_SIZE)

tokenizer.pre_tokenizer = Whitespace()

files = [ ALL_SETENCES_FILE ]

tokenizer.train(files, bpe_trainer)

In [8]:
# постпроцессор будет отвечать за добавление спец токенов в начало и в конец предложения
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

tokenizer.save("tokenizer.json")

In [9]:
tokenizer.encode('[BOS]')

Encoding(num_tokens=3, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [10]:
# PreTrainedTokenizerFast -- это обертка над предобученным токенизатором, которой мы будем дальше пользоваться
fast_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
fast_tokenizer.bos_token = "[BOS]"
fast_tokenizer.eos_token = "[EOS]"
fast_tokenizer.pad_token = "[PAD]"
fast_tokenizer.unk_token = "[UNK]"

In [11]:
fast_tokenizer.save_pretrained("rnn_tokenizer")

('rnn_tokenizer/tokenizer_config.json',
 'rnn_tokenizer/special_tokens_map.json',
 'rnn_tokenizer/tokenizer.json')

In [12]:
# пример работы токенизатора
fast_tokenizer("test sentence", text_target="phrase cible")

{'input_ids': [1, 5287, 4566, 2], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1], 'labels': [1, 5573, 13780, 2]}

In [13]:
source_lang = "en"
target_lang = "fr"

def preprocess_function(examples):
    inputs = [example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = fast_tokenizer(inputs, text_target=targets, max_length=64, truncation=True, add_special_tokens=True)
    return model_inputs


In [14]:
# токенизируем датасет
books_preprocessed = books.map(preprocess_function, batched=True)

Map:   0%|          | 0/7135 [00:00<?, ? examples/s]

Map:   0%|          | 0/1784 [00:00<?, ? examples/s]

In [15]:
list(books_preprocessed['train'][0].keys())

['id',
 'translation',
 'input_ids',
 'token_type_ids',
 'attention_mask',
 'labels']

In [16]:
from transformers import DataCollatorForSeq2Seq

# DataCollator отвечает за объединение данных в батчи -- добивает предложения до одной длинны (делает паддинг)
# преобразует numpy.array или питоновские списки в torch.Tensor
data_collator = DataCollatorForSeq2Seq(tokenizer=fast_tokenizer, return_tensors="pt")

In [17]:
fast_tokenizer("test sentence", text_target="phrase cible")

{'input_ids': [1, 5287, 4566, 2], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1], 'labels': [1, 5573, 13780, 2]}

In [18]:
# пример работы DataCollator
# обратите внимание, что входные данные были разной длинны
data_collator( [ {"input_ids": [ 100, 200, 300 ], "labels": [100,200,300, 400, 500]}, { "input_ids": [ 100, 200, 300,400,500], "labels": [100,200,300,400,500,600]} ] )


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'input_ids': tensor([[100, 200, 300,   3,   3],
        [100, 200, 300, 400, 500]]), 'labels': tensor([[ 100,  200,  300,  400,  500, -100],
        [ 100,  200,  300,  400,  500,  600]]), 'attention_mask': tensor([[1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1]])}

In [19]:
# ВАЖНО! Эту ячейку в пайплайнах будет копировать в отдельный файл.
# Нельзя изменять названия и конструкторы классов.
# Не стоит изменять сигнатуры методов
# Не стоит добавлять сюда новые ненужные импорты
# Не стоит удалять импорты, которые тут были
# Не стоит добавлять в эту ячейку новые классы

from transformers.modeling_outputs import Seq2SeqModelOutput, Seq2SeqLMOutput

from transformers import GenerationConfig, PretrainedConfig, PreTrainedModel
import random

import torch
import torch.nn as nn

# в этой ячейке будем пилить реализацию самой RNN'ки
# у пайторча есть релевантный туториал, в котором тоже решают задачу машинного перевода
# https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html
##
# для прохождения тестов не будет требоваться, чтобы модель полностью обучилась
# это может занять много времени. В этой домашке будет достаточно переобучить модель
# на небольшой части датасета.
# 
# Цель домашки -- это поработать с RNN'ками, MultiHeadAttention
# Понять, как люди жили до изобретения трансформеров
# Нужно дополнить процесс обучения ниже, заполнить пропуски
# 
# Чтобы домашка была тестируемой, пришлось прибегнуть к таким требованиям
# Кроме того, нет большого практического смысла в обучении такой архитектуры,
# потому что она устарела, лучше обучите трансформер во втором ноутбуке)
# 
# HINT!
# Прописывайте размерности разных тензоров в комментах -- так будет проще разбираться в коде и дебажить
# 

# PretrainedConfig см тут https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/configuration#transformers.PretrainedConfig
class RNNAttentionConfig(PretrainedConfig):
    model_type = "rnn"

    r"""
    В классе конфига должны быть описаны все гипер-параметры модели

    Args:
        vocab_size (`int`):
            Размер словаря
        embedding_dim (`int`):
            Размерность эмбэддингов
        hidden_dim (`int`):
            размерность скрытых слоев
        num_layers (`int`):
            количество слоев для GRU (и для енкодера, и для декодера)
        teacher_forcing_proba (`float`, *optional*, defaults to 0.2):
            teacher forcing -- это способ ускорить обучение RNN модели
            Эта уловка должна помогать, тк это подсказка для
            RNN'ки для генерации более правильной последовательности.
            Суть заключается в том, что с определенной вероятностью, мы будем передавать
            На вход RNN'ке не предсказанный моделью токен, а токен из целевой последовательности.
        max_length (`int`):
            Максимальная длинна сгенерированной последовательности

    """

    def __init__(
        self,
        vocab_size=20000,
        embedding_dim=128,
        hidden_dim=384,
        num_layers=2,
        teacher_forcing_proba=0.2,
        max_length=64,
        # Эти токены должны быть предопределены уже в PretrainedConfig
        # https://github.com/huggingface/transformers/blob/cf11493dce0a1d22446efe0d6c4ade02fd928e50/src/transformers/configuration_utils.py#L214
        # pad_token_id=None,
        # bos_token_id=None,
        # eos_token_id=None,
        **kwargs,
    ):
        super().__init__(**kwargs, max_length=max_length)

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.teacher_forcing_proba = teacher_forcing_proba


# вот тут про PreTrainedModel  https://github.com/huggingface/transformers/blob/cf11493dce0a1d22446efe0d6c4ade02fd928e50/src/transformers/modeling_utils.py#LL1009C7-L1009C22
class Seq2SeqRNNAttention(PreTrainedModel):

    config_class = RNNAttentionConfig
    supports_gradient_checkpointing = False


    def __init__(self, config):
        super().__init__( config )

        self.generation_config = GenerationConfig()
        self.generation_config.max_length = config.max_length

        ## Задача 1:
        # правильно расставить гиперпараметры для всех
        # модулей в классе.
        # Что может пригодиться:
        # self.config.vocab_size, self.config.embedding_dim
        # self.config.hidden_dim, self.config.num_layers

        self.embeddings = nn.Embedding(num_embeddings="TODO", embedding_dim="TODO")
        self.embeddings_dropout = nn.Dropout(p=0.1)

        # если не делать нормализацию, в RNN'ках (да и в других типах сетей)
        # можно столкнуться с NaN'ами в лоссе, это очень неприятно
        # но на всякий случай, есть штука, которая поможет дебажить такие кейсы
        # https://pytorch.org/docs/stable/autograd.html#anomaly-detection
        self.encoder_norm = nn.LayerNorm("TODO")

        # дока по интерфесу тут https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
        self.encoder = nn.GRU("TODO", "TODO", "TODO", batch_first=True)

        # опишем декодер
        self.decoder_norm = nn.LayerNorm("TODO")
        self.decoder_hidden_norm = nn.LayerNorm("TODO")
        self.decoder = nn.GRU("TODO", "TODO", "TODO", batch_first=True)

        self.decoder_attention_norm = nn.LayerNorm("TODO")
        self.decoder_attention = nn.MultiheadAttention(embed_dim="TODO", num_heads=8, batch_first=True)

        # линейное преобразование для предсказания вероятностей каждого токена
        self.decoder_labels_linear = nn.Linear("TODO", "TODO")

        # nn.CrossEntropyLoss -- это nn.Softmax + nn.NLLLoss, но более вичислительно стабильный и оптимальный
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id)

        return

    def encode(self, input_embeddings=None):
        """
        В этом методе вам нужно получить контекстные эмбэддинги исходной последовательности с помощью GRU
        """

        batch_size = input_embeddings.size(0)

        input_embeddings = self.encoder_norm(input_embeddings)

        # инициализируем скрытое состояние рандомом
        hidden_state = torch.rand((self.config.num_layers, batch_size, self.config.hidden_dim), device=input_embeddings.device) / 100
        rnn_outputs, last_hidden_state = self.encoder(input_embeddings, hidden_state)

        return rnn_outputs, last_hidden_state

    def decode(self, encoder_outputs=None, encoder_last_hidden_state=None, encoder_key_padding_mask=None, target_labels=None):
        """
        Этот метод используется при обучении, вызывается в forward
        В этом методе ничего не придется править, скорее всего.
        """
        decoder_hidden_state = encoder_last_hidden_state

        decoder_hidden_state = self.decoder_hidden_norm(decoder_hidden_state)

        max_length = target_labels.shape[1] - 1
        batch_size = target_labels.shape[0]
        _, decoder_output = self.generate_encoded(
            batch_size=batch_size,
            encoder_outputs=encoder_outputs,
            encoder_key_padding_mask=encoder_key_padding_mask,
            encoder_last_hidden_state=decoder_hidden_state,
            max_length=max_length,
            target_labels=target_labels
        )
        decoder_output = torch.cat(decoder_output, dim=1)

        return decoder_output, decoder_hidden_state

    # https://huggingface.co/docs/transformers/v4.28.1/en/main_classes/output#transformers.modeling_outputs.Seq2SeqModelOutput
    def forward(self, input_ids=None, labels=None, attention_mask=None, token_type_ids=None) -> Seq2SeqModelOutput:
        """
        Метод для процесса обучения, который должен возвращать объект Seq2SeqModelOutput
        """
        
        # пример, как прописать размерность для переменной
        # input_ids [ batch_size, seq_len ]
        # labels    [ batch_size, seq_len ]

        encoder_key_padding_mask = (attention_mask == 0)

        input_embeddings = self.embeddings(input_ids) # TODO какая размерность [ ? ]
        input_embeddings = self.embeddings_dropout(input_embeddings)

        labels[labels == -100] = self.config.pad_token_id

        # TODO размерность encoder_outputs           [ ? ]
        # TODO размерность encoder_last_hidden_state [ ? ]
        encoder_outputs, encoder_last_hidden_state = "TODO" # TODO запускаем Энкодер с нужными аргументами

        decoder_outputs, decoder_hidden_state = "TODO" # TODO запускаем Денкодер с нужными аргументами

        # надо получить логиты для вероятностей 
        # целевая размерность: [ bs, target_seq_len - 1, vocab_size ]
        labels_probas = "TODO"

        # Считаем лосс
        # ВАЖНО! Декодер должен генерировать только target_seq_len - 1 токенов
        # тк он предсказывает только последующий токен и токен BOS ему предсказывать из "ничего" нельзя
        # поэтому таргет тоже должен быть смещен
        # внимательно ознакомьтесь с документацией CrossEntropyLoss
        # https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
        loss = self.criterion( "TODO" )

        return Seq2SeqLMOutput(
            loss=loss,
            past_key_values=None,
            decoder_hidden_states=decoder_outputs,
            encoder_last_hidden_state=encoder_last_hidden_state,
            encoder_hidden_states=encoder_outputs,
        )

    def generate_encoded(self, batch_size=None, encoder_outputs=None, encoder_key_padding_mask=None, encoder_last_hidden_state=None, max_length=None, target_labels=None, **kwargs):
        """
        Генерирует последовательность токенов из энкоженых эмбэддингов исходной последовательности

        В этом методе надо реализовать жадный алгоритм при генерации последующих токенов
        """

        last_tokens_batch = torch.tensor([self.config.bos_token_id] * batch_size, device=encoder_last_hidden_state.device).unsqueeze(1) # [ bs, 1 ]


        # последовательность предсказанных токенов по батчам
        # каждый элемент в массиве -- это предсказанный токен на позиции `i`
        # для всех батчей
        predicted_tokens_sequences = [ last_tokens_batch ] # result shape [ bs, max_seq_len ]

        # todo размерность decoder_outputs: [ ? ]
        decoder_outputs = [] # тут будем хранить output'ы декодера по всему циклу

        softmax_layer = nn.Softmax(dim=-1) # используем для получения вероятностей из логитов для предсказания следующих токенов

        decoder_hidden_state = encoder_last_hidden_state

        if target_labels is not None:
            # если передали target_labels, это значит, что вызов метода был из
            # self.decode() во время обучения и сгенерировать надо tgt_seq_len - 1 токенов
            max_length = target_labels.shape[1] - 1

        if max_length is None:
            max_length = self.generation_config.max_length

        for token_i in range(max_length):
            # greedy decode

            # last_tokens_embeddings: [bs, 1, embedding_dim]
            if random.random() < self.config.teacher_forcing_proba and target_labels is not None:
                last_tokens_embeddings = # берем из таргета
            else:
                last_tokens_embeddings = # берем из того, что сгенерили на предыдущей итерации
            
            lebels_embeddings_norm = self.decoder_norm(last_tokens_embeddings)
            decoder_hidden_state = self.decoder_hidden_norm(decoder_hidden_state)

            # decoder_output: [ batch_size, 1, hidden_dim ]
            decoder_output, decoder_hidden_state = "TODO" # запускаем декодер

            # применяем аттеншн, не забываем про параметр key_padding_mask
            decoder_output_with_attention, _ = "TODO"

            # нормируем выход аттеншна + делаем резидуал connection как в трансформерах
            decoder_output_with_attention_with_residual = decoder_output_with_attention + decoder_output
            decoder_output_with_attention = self.decoder_attention_norm(decoder_output_with_attention_with_residual)

            decoder_outputs.append(decoder_output_with_attention)

            # TODO размерность?
            next_token_logits = "TODO"
            
            # из логитов надо получить вероятности токенов (выше перед циклом есть подсказка)
            # и тк декодер у нас жадный, вам нужен будет вот этот метод
            # https://pytorch.org/docs/stable/generated/torch.max.html
            last_tokens_batch = "TODO"

            predicted_tokens_sequences.append(last_tokens_batch)

        # Нужно вернуть тензор predicted_tokens_sequences
        # см torch.cat https://pytorch.org/docs/stable/generated/torch.cat.html
        predicted_tokens_sequences = "TODO"

        # теперь для каждого батча, если сеть сгенерила EOS
        # мы явно пропишем все последующие токены = pad_token_id
        # тк EOS -- это последний токен. Хотя в идеале, сеть сама должна это выучить
        eos_token_indexes = torch.nonzero(predicted_tokens_sequences == self.config.eos_token_id, as_tuple=False)
        for eos_idx in eos_token_indexes:
            predicted_tokens_sequences[eos_idx[0], eos_idx[1]:] = self.config.pad_token_id

        return predicted_tokens_sequences, decoder_outputs

    def generate(self, input_ids=None, labels=None, attention_mask=None, token_type_ids=None, max_length=None, **kwargs):
        """
        Метод используется для генерации целевой последовательности
        из последовательности исходных токенов
        Будет вызываться из trainer'а на этапе evaluate
        """
        
        batch_size = input_ids.shape[0]

        encoder_key_padding_mask = None
        if attention_mask is not None:
            encoder_key_padding_mask = (attention_mask == 0)

        # 1. получаем эмбэддинги
        # 2. Запускаем енкодер
        # 3. Запускаем generate_encoded

        # вернуть надо predicted_tokens_sequences
        return predicted_tokens_sequences




In [20]:
# проверяем, что модель не падает на генерации

rnn_config = RNNAttentionConfig(
    pad_token_id=fast_tokenizer.pad_token_id,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
)

rnn_attention_model = Seq2SeqRNNAttention(rnn_config)
rnn_attention_model.generate( input_ids = torch.arange(15).reshape(3, 5), max_length=64 )

tensor([[    1, 14910, 14910, 15235, 15235, 15235,  3650,  3650,  3650,  3650,
          3650,  3650,  3650, 19797, 19797, 19797, 19797, 19797,  3113, 12574,
         12574, 12574,  3984,  3984,  3984,  3984,  3984,  3984,  3984,  9739,
          9210,  9210,  9210,  9210,  9210,  9210,  9210,  9210,  9210,  9210,
          9210,  4096, 13495, 13495, 13495, 13495, 13495, 13495, 13495, 13495,
         19693, 19693, 19693, 13456, 13456, 13456, 13456, 13456, 13456, 13456,
         13456, 13456, 18605,  6092,  6092],
        [    1, 16802, 16802,  4348,  4348,  4348,  4348, 14120, 14120, 15003,
         15003, 15003, 15003, 15003, 15003, 15003, 19542, 19542, 11322, 13157,
         13157, 13157, 13157, 13157, 13157, 13157, 13157, 13157, 13157, 13157,
         13157, 16818, 16818, 18671, 14185, 14185, 14185, 14185, 19439, 19439,
         10469, 10469, 15156, 15156, 15156, 10469, 10469, 18671, 16417, 16417,
         16417, 18671, 16417,  5568,  5568,  5568,  5568,  5568,  5568,  5568,
       

In [21]:
# оценим кол-во параметров модели
sum( p.numel() for p in rnn_attention_model.parameters() if p.requires_grad )

14403872

In [22]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [23]:
import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = fast_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, fast_tokenizer.pad_token_id)
    decoded_labels = fast_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # хочется смотреть глазами на то, что генерит сетка, самый простой способ -- это напечатать принтами
    # лучше печатать со спец токенами, в некоторых случаях это поможет оттебажить и найти ошибку
    print('Predicted decoded_preds[0]:', fast_tokenizer.batch_decode(preds, skip_special_tokens=False)[0])
    print('Target decoded_labels[0]  :', decoded_labels[0])

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != fast_tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}

    return result

In [31]:

rnn_config = RNNAttentionConfig(
    pad_token_id=fast_tokenizer.pad_token_id,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
    teacher_forcing_proba=0.2
)

rnn_attention_model = Seq2SeqRNNAttention(rnn_config)

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_opus_books_model",
    evaluation_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=100,
    predict_with_generate=True,
    logging_steps=100,
)

trainer = Seq2SeqTrainer(
    model=rnn_attention_model,
    args=training_args,
    train_dataset=books_preprocessed["train"].select(range(128)),              # берем только часть данных для переобучения
    eval_dataset=books_preprocessed["train"].select(torch.tensor(range(128))),  # валидировать будем тоже на обучающих данных (дисклаймер: это можно делать только для тестирования)
    tokenizer=fast_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [32]:
# Адекватное время тренировки для этой задачки - не более 5 минут
# Если у вас получается сильно больше или не получается выбить
# необходимый скор, скорее всего, где-то ошибка
trainer.train()

Step,Training Loss,Validation Loss,Bleu,Gen Len
1000,0.626900,0.462908,85.715900,25.007800


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/activation.py:1144: UserWarning: Converting mask without torch.bool dtype to bool; this will negatively affect performance. Prefer to use a boolean mask directly. (Triggered internally at ../aten/src/ATen/native/transformers/attention.cpp:150.)
  return torch._native_multi_head_attention(


Predicted decoded_preds[0]: [BOS] Meaulnes, sur la prière de Mlle de Galais, est maintenant très aimable avec lui. [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
Target decoded_labels[0]  : ['Meaulnes, sur la prière de Mlle de Galais, est maintenant très aimable avec lui.']


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <cell line: 2>:2                                                                              │
│                                                                                                  │
│ /usr/local/lib/python3.10/dist-packages/transformers/trainer.py:1664 in train                    │
│                                                                                                  │
│   1661 │   │   inner_training_loop = find_executable_batch_size(                                 │
│   1662 │   │   │   self._inner_training_loop, self._train_batch_size, args.auto_find_batch_size  │
│   1663 │   │   )                                                                                 │
│ ❱ 1664 │   │   return inner_training_loop(                                                       │
│   1665 │   │   │   args=args,                                                                    │
│   1666 │   │   │   resume_from_checkpoint=resume_from_checkpoint,                                │
│   1667 │   │   │   trial=trial,                                                                  │
│                                                                                                  │
│ /usr/local/lib/python3.10/dist-packages/transformers/trainer.py:1940 in _inner_training_loop     │
│                                                                                                  │
│   1937 │   │   │   │   │   with model.no_sync():                                                 │
│   1938 │   │   │   │   │   │   tr_loss_step = self.training_step(model, inputs)                  │
│   1939 │   │   │   │   else:                                                                     │
│ ❱ 1940 │   │   │   │   │   tr_loss_step = self.training_step(model, inputs)                      │
│   1941 │   │   │   │                                                                             │
│   1942 │   │   │   │   if (                                                                      │
│   1943 │   │   │   │   │   args.logging_nan_inf_filter                                           │
│                                                                                                  │
│ /usr/local/lib/python3.10/dist-packages/transformers/trainer.py:2753 in training_step            │
│                                                                                                  │
│   2750 │   │   │   # loss gets scaled under gradient_accumulation_steps in deepspeed             │
│   2751 │   │   │   loss = self.deepspeed.backward(loss)                                          │
│   2752 │   │   else:                                                                             │
│ ❱ 2753 │   │   │   loss.backward()                                                               │
│   2754 │   │                                                                                     │
│   2755 │   │   return loss.detach()                                                              │
│   2756                                                                                           │
│                                                                                                  │
│ /usr/local/lib/python3.10/dist-packages/torch/_tensor.py:487 in backward                         │
│                                                                                                  │
│    484 │   │   │   │   create_graph=create_graph,                                                │
│    485 │   │   │   │   inputs=inputs,                                                            │
│    486 │   │   │   )                                                                             │
│ ❱  487 │   │   torch.autograd.backward(                                                          │
│    488 │   │   │   self, gradient, retain_graph, create_gra

In [33]:
rnn_attention_model.save_pretrained("./rnn_attention_model", safe_serialization=False)

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir my_awesome_opus_books_model/runs

## Протестируем модель

In [34]:
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

# DataCollator отвечает за объединение данных в батчи -- добивает предложения до одной длинны (делает паддинг)
# преобразует numpy.array или питоновские списки в torch.Tensor

import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

loaded_tokenizer = AutoTokenizer.from_pretrained("./rnn_tokenizer/")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = loaded_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, loaded_tokenizer.pad_token_id)
    decoded_labels = loaded_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != loaded_tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


test_rnn_config = RNNAttentionConfig(
    pad_token_id=loaded_tokenizer.pad_token_id,
    bos_token_id=loaded_tokenizer.bos_token_id,
    eos_token_id=loaded_tokenizer.eos_token_id,
)

rnn_attention_model_loaded = Seq2SeqRNNAttention.from_pretrained("./rnn_attention_model")

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_opus_books_model",
    evaluation_strategy="steps",
    eval_steps=100,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=100,
    predict_with_generate=True,
    logging_steps=100,
)


data_collator = DataCollatorForSeq2Seq(tokenizer=loaded_tokenizer, return_tensors="pt")

trainer = Seq2SeqTrainer(
    model=rnn_attention_model_loaded,
    args=training_args,
    eval_dataset=books_preprocessed["train"].select(torch.tensor(range(128))),  # валидировать будем тоже на обучающих данных (дисклаймер: это можно делать только для тестирования)
    tokenizer=loaded_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

evaluate_result = trainer.evaluate(test_dataset=books_preprocessed["train"].select(range(128)))

print("evaluate_result", evaluate_result)

assert evaluate_result['eval_bleu'] >= 50

You are using a model of type rnn to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/activation.py:1144: UserWarning: Converting mask without torch.bool dtype to bool; this will negatively affect performance. Prefer to use a boolean mask directly. (Triggered internally at ../aten/src/ATen/native/transformers/attention.cpp:150.)
  return torch._native_multi_head_attention(


Если ассерт выше прошел, можете загрузить ноутбук и обученные модельки+конфиги на гитхаб. На гитхабе из ноутбука будет выгружена ячейка с описанием модели и будет запускаться тест аналогичный ячейке выше.

Веса и конфиги надо заархивировать и на гитхаб загрузить в виде архива с сохранением названий: `rnn_tokenizer.zip`, `rnn_attention_model.zip`

Ноутбук надо загрузить с таким же названием, какое было в репозитории.

In [35]:
!zip -r rnn_tokenizer.zip rnn_tokenizer

  adding: rnn_tokenizer/ (stored 0%)


In [36]:
!zip -r rnn_attention_model.zip rnn_attention_model

  adding: rnn_attention_model/ (stored 0%)


Мы зааихивировали конфиги и веса, скачайте архивы из колаба и загрузите в github репозиторий. **Не забудьте обновить ноутбук в github репозитории тоже!**


Архив с моделькой может занимтаь больше 25 мегабайт. Гитхаб не разрешает грузить большие файлы через веб-интерфейс. Можете закоммитить эти архивы через консоль или погуглить, как это сделать по-другому
https://bytesbin.com/upload-files-larger-than-25mb-to-github/
https://www.google.com/search?q=Fix+GitHub+%E2%80%98Yowza+That%E2%80%99s+a+Big+File%E2%80%99

# Можно протестировать нашу обученную модель в заранее подготовленном пайплайне от huggingface

In [37]:
from transformers import pipeline

In [38]:
en_fr_translator = pipeline("translation_en_to_fr", model=rnn_attention_model, tokenizer=fast_tokenizer, device=torch.device("cuda:0"))
en_fr_translator

The model 'Seq2SeqRNNAttention' is not supported for translation_en_to_fr. Supported models are ['BartForConditionalGeneration', 'BigBirdPegasusForConditionalGeneration', 'BlenderbotForConditionalGeneration', 'BlenderbotSmallForConditionalGeneration', 'EncoderDecoderModel', 'FSMTForConditionalGeneration', 'GPTSanJapaneseForConditionalGeneration', 'LEDForConditionalGeneration', 'LongT5ForConditionalGeneration', 'M2M100ForConditionalGeneration', 'MarianMTModel', 'MBartForConditionalGeneration', 'MT5ForConditionalGeneration', 'MvpForConditionalGeneration', 'NllbMoeForConditionalGeneration', 'PegasusForConditionalGeneration', 'PegasusXForConditionalGeneration', 'PLBartForConditionalGeneration', 'ProphetNetForConditionalGeneration', 'SwitchTransformersForConditionalGeneration', 'T5ForConditionalGeneration', 'XLMProphetNetForConditionalGeneration'].


In [39]:
books_preprocessed["train"][0]

{'id': '2520',
 'translation': {'en': 'Meaulnes, at the request of Mademoiselle de Galais, is now very nice to him.',
  'fr': 'Meaulnes, sur la prière de Mlle de Galais, est maintenant très aimable avec lui.'},
 'input_ids': [1,
  346,
  11,
  127,
  121,
  5660,
  145,
  2204,
  119,
  853,
  11,
  118,
  316,
  321,
  7391,
  131,
  265,
  13,
  2],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [1,
  346,
  11,
  259,
  129,
  7465,
  119,
  1083,
  119,
  853,
  11,
  209,
  1072,
  1232,
  2381,
  314,
  289,
  13,
  2]}

In [40]:
en_fr_translator(books_preprocessed["train"][0]["translation"]['en'])

[{'translation_text': 'Meaulnes , sur la prière de Mlle de Galais , est maintenant très aimable avec lui .'}]

Тк мы переобучили модель, то на данных, которых не было в обучающем датасете, наша модель не может выдать что-то осмысленное(

In [44]:
en_fr_translator("How are you?")

[{'translation_text': ''}]